# New fastai text pipeline

## TODO:
- remove HF_TOKEN warnings
- format show_batch to LM dataloaders to 2 columns table

We’re rewriting fastai’s text pipeline with two main changes:

- Replacing spaCy’s word tokenizer with HF tokenizers (defaulting to GPT-2)
- Adding native support for HF Datasets

This new API lets users quickly create `TextDataLoaders` when the dataset follows standard patterns (shown below), while remaining flexible when the dataset doesn't follow those patterns or users want to inspect the pipeline step by step (getting documents, labels, splits, etc.).

We’ll create dataloaders from datasets organized in 5 different ways. The first two are for language modelling and the last three are for classification.

- **[answerdotai/simplewiki](https://huggingface.co/datasets/answerdotai/simplewiki)**: HF dataset with one doc per row
- **[Salesforce/wikitext](https://huggingface.co/datasets/Salesforce/wikitext)**: HF dataset that requires merging rows into docs
- **[stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb)**: HF classification dataset
- **fastai IMDb**: local classification dataset following a `split/label/file` layout
- **Custom data**: custom dataset built with the mid-level API

In [ ]:
from fastai.text.all import *
from fastai.text.tokenizer import *

## SimpleWiki

In this section, we will load the [HF SimpleWiki dataset](https://huggingface.co/datasets/answerdotai/simplewiki). The dataset has one doc per row and works out of the box with the `load_dataset` function.

In [ ]:
from datasets import load_dataset


First, let’s load the `articles` subset and look at the first three rows of the training split.

In [ ]:
wiki_sample = load_dataset('answerdotai/simplewiki', 'articles', split='train[:3]')
wiki_sample.to_pandas()


,article_id,page_id,revision_id,title,timestamp,md,warnings
0,0,1,1806169,April,2026-05-18T12:42:53Z,"**April** (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.\n\n## The Month\n\nApril comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.\n\nApril begins on the same day of the week as Jul...",[]
1,1,2,1806169,August,2026-05-18T12:48:35Z,"**August** (Aug.) is the eighth month of the year in the Gregorian calendar, coming between July and September. It has 31 days. It is named after the Roman emperor Augustus Caesar.\n\nAugust does not begin on the same day of the week as any other month in common years, but begins on the same day of the week as February in leap years. August always ends on the same day of the week as November.\n\n## The Month\n\nThis month was first called *Sextilis* in Latin, because it was the sixth month in the old Roman calendar. The Roman calendar began in March about 735 BC with Romulus. October was t...",[]
2,2,6,693482,Art,2026-08-03T13:45:35Z,"**Art** is a creative activity. It produces a product, an object. **Art** is a diverse range of human activities in creating visual, performing subjects, and expressing the author's thoughts. The product of art is called a **work of art**, for others to experience.\n\nSome art is useful in a practical sense, such as a sculptured clay bowl that can be used. That kind of art is sometimes called a *craft*.\n\nThose who make art are called artists. They hope to affect the emotions of people who experience it. Some people find art relaxing, exciting or informative. Some say people are driven to...",[]


Each row contains one complete document in the `md` column, so we can create `TextDataLoaders` directly from this dataset. To keep tokenization quick, we’ll use only a small portion of the training split. We can do this by selecting a slice of the data with `train='train[:80]', valid='train[80:100]'`.

In [ ]:
simple_dls = TextDataLoadersNew.from_dataset('answerdotai/simplewiki', subset='articles',
    text_col='md', train='train[:80]', valid='train[80:100]',
    bs=4, seq_len=32)

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Let’s use `show_batch()` to check that the text looks right. We can see that fastai adds the special tokens `<bos>` and `<eos>` to mark where each document starts and ends.

In [ ]:
simple_dls.show_batch()

<bos>A **church** is a building that was constructed to allow people to meet to worship together. These people are usually Christians, or influenced by Christianity. Some
 supervised by the bankruptcy court and usually lasts for three to five years. Businesses cannot file for Chapter 13 bankruptcy.

### Other bankruptcy chapters

Less
 the roar of rushing waters."

Ezekiel 10:5-7 ; Ezekiel 10:8 reveals that they have hands like a man under their wings
 like Ascanio Sobrero and Alfred Nobel developed powerful materials such as nitroglycerin and dynamite. These explosives were used in mining, building tunnels,
A **church** is a building that was constructed to allow people to meet to worship together. These people are usually Christians, or influenced by Christianity. Some other
 by the bankruptcy court and usually lasts for three to five years. Businesses cannot file for Chapter 13 bankruptcy.

### Other bankruptcy chapters

Less common
 roar of rushing waters."

Ezekiel 10:5-7 ; Ezekie

`show_batch()` lets us read the text, but the model actually receives token IDs. We can use `one_batch()` to check the exact input and target tensors.

In [ ]:
one_batch = simple_dls.one_batch()

In [ ]:
one_batch[0]

LMTensorText([[50258,    32, 12429, 33215,  1174,   318,   257,  4572,   326,
                3544, 17075,   284,  5128,    11,  1429,    11,  3650,    11,
                 290,  5072,  1366,    13,  6060,   318,  1321,   884,   355,
                3146,    11,  2456,    11,   290],
              [ 7514, 11056,    13,  1119,  2291, 27997,    11, 18431,   349,
                 541,  2340,    11,   290, 31397,    13, 24701,  2340,   466,
                 407,  5022,   880,   351,  1660,   780,   484,   389,  4632,
                1729,    79,  6192,   290,  7409],
              [40719, 31944,   331,    89,    59,   737,   198,   198,   818,
               37139,    11, 48816, 16792,    88, 22725,   416, 16792,    89,
               22725,   357,   273, 16792,    88, 22725,  1661, 16792,    89,
                  59,  4008,   460,   307,  3194],
              [ 6109,   513,   812,    11,  4505,  6622,   663,  3071,   543,
                 318,   618,   257,  3381,   287,   262,  2156,   2

We can use `tok.decode` to turn these IDs back to text.

In [ ]:
simple_dls.tok.decode(one_batch[0][0].tolist(), skip_special_tokens=False)

'<bos>A **computer** is a machine that uses electronics to input, process, store, and output data. Data is information such as numbers, words, and'

In [ ]:
#TODO: Can we avoid tolist when running tok.decode?

## WikiText-103

In this section, we’ll work with the [WikiText-103 dataset](https://huggingface.co/datasets/Salesforce/wikitext). Its format isn’t quite what we need: each document is spread across several rows. We need to merge the rows between article headings (`= Title =`) to form a complete document.

In [ ]:
wikitext_sample = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train[:8]')
wikitext_sample.to_pandas()

,text
0,
1,= Valkyria Chronicles III = \n
2,
3,"Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the "" Nameless "" , a penal military unit serving the nation of Gallia during the Second E..."
4,"The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . \n"
5,"It met with positive sales in Japan , and was praised by both Japanese and western critics . After release , it received downloadable content , along with an expanded edition in November of that year . It was also adapted into manga and an original video animation series . Due to low sales of Valkyria Chronicles II , Valkyria Chronicles III was not localized , but a fan translation compatible with the game 's expanded edition was released in 2014 . Media.Vision would return to the franchise with the development of Valkyria : Azure Revolution for the PlayStation 4 . \n"
6,
7,= = Gameplay = = \n


The `get_wiki103_data` function below is the one responsible for merging rows into documents. It loops through all rows in the HF dataset and groups rows between article headings (`= Title =`) to form complete documents.

In [ ]:
def get_wiki103_data(ds):
    "Assemble WikiText-103 articles from source lines"
    heading, docs, cur = re.compile(r'^ = [^=].* = \n$'), [], []
    for text in ds['text']:
        if heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    texts = [d for d in (*docs, ''.join(cur)) if d.strip()]
    return texts, pd.DataFrame(index=range(len(texts)))

We pass `get_wiki103_data` to the `get_data` parameter of `TextDataLoaders.from_dataset`.

In [ ]:
wiki_dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:3000]', valid='validation',
    bs=4, seq_len=32)

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [ ]:
wiki_dls.show_batch()

<bos> = 2011 – 12 Columbus Blue Jackets season = 
 The 2011 – 12 Columbus Blue Jackets season was the team 's 12th season in the National Hockey
 in a Bulls win . The Bulls were fined $ 30 @,@ 000 for the game : $ 25 @,@ 000 for failing to report the impromptu
 on herding cattle , with people living a nomadic or semi @-@ nomadic way of life . Although witnessing some land clearance , Britain was largely forest
 his representative in the region . However , Zapata was worried that if he did not fully control all the major towns in Morelos by the time that Madero
 = 2011 – 12 Columbus Blue Jackets season = 
 The 2011 – 12 Columbus Blue Jackets season was the team 's 12th season in the National Hockey League
 a Bulls win . The Bulls were fined $ 30 @,@ 000 for the game : $ 25 @,@ 000 for failing to report the impromptu number
 herding cattle , with people living a nomadic or semi @-@ nomadic way of life . Although witnessing some land clearance , Britain was largely forested
 representativ

## IMDb: classification

In this section, we create classification dataloaders using the [stanfordnlp/imdb](https://huggingface.co/datasets/stanfordnlp/imdb) dataset.

In [ ]:
imdb_sample = load_dataset('stanfordnlp/imdb', split='train[:8]')
imdb_sample.to_pandas()

,text,label
0,"I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered ""controversial"" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain politica...",0
1,"""I Am Curious: Yellow"" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies li...",0
2,"If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />",0
3,"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the dif...",0
4,"Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee song..<br /><br />""Is that all there is??"" ...I was just an early teen when this smoked fish hit the U.S. I was too young to get in the theater (although I did manage to sneak into ""Goodbye Columbus""). Then a screening at a local film museum beckoned - Finally I could see this film, except now I was as old as my parents were when they schlepped to see it!!<br /><br />The ONLY reason this film was not condemned to the anonymous sands of time was because of the obscenity case spa...",0
5,"I would put this at the top of my list of films in the category of unwatchable trash! There are films that are bad, but the worst kind are the ones that are unwatchable but you are suppose to like them because they are supposed to be good for you! The sex sequences, so shocking in its day, couldn't even arouse a rabbit. The so called controversial politics is strictly high school sophomore amateur night Marxism. The film is self-consciously arty in the worst sense of the term. The photography is in a harsh grainy black and white. Some scenes are out of focus or taken from the wrong angle. ...",0
6,"Whoever wrote the screenplay for this movie obviously never consulted any books about Lucille Ball, especially her autobiography. I've never seen so many mistakes in a biopic, ranging from her early years in Celoron and Jamestown to her later years with Desi. I could write a whole list of factual errors, but it would go on for pages. In all, I believe that Lucille Ball is one of those inimitable people who si

The only difference from the previous language model sections is that we need to specify the label. We pass the label column name to the `label_col` parameter.

In [ ]:
imdb_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:50]+train[-50:]', valid='test[:20]+test[-20:]',
    bs=4, seq_len=32)


 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Now `show_batch()` shows each text with its label.

In [ ]:
imdb_clas.show_batch()

,text,category
0,"<bos>I have this film out of the library right now and I haven't finished watching it. It is so bad I am in disbelief. Audrey Hepburn had totally lost her talent by then, although she'd pretty much finished with it in 'Robin and Marian.' This is the worst thing about this appallingly stupid film. It's really only of interest because it was her last feature film and because of the Dorothy Stratten appearance just prior to her homicide.<br /><br />There is nothing but idiocy between Gazzara and his cronies. Little signals and little bows and nods to real screwball comedy of which this is the faintest, palest shadow.<br /><br />Who could believe that there are even some of the same Manhattan environs that Hepburn inhabited so magically and even mythically in 'Breakfast at Tiffany's' twenty years earlier? The soundtrack of old Sinatra songs and the Gershwin song from which the",0
1,"<bos>WARNING: This review contains SPOILERS. Do not read if you don't want some points revealed to you before you watch the film.<br /><br />With a cast like this, you wonder whether or not the actors and actresses knew exactly what they were getting into. Did they see the script and say, `Hey, Close Encounters of the Third Kind was such a hit that this one can't fail.' Unfortunately, it does. Did they even think to check on the director's credentials? I mean, would YOU do a movie with the director of a movie called `Satan's Cheerleaders?' Greydon Clark, who would later go on to direct the infamous `Final Justice,' made this. It makes you wonder how the people of Mystery Science Theater 3000 could hammer `Final Justice' and completely miss out on `The Return.'<br /><br />The film is set in a small town in New Mexico. A little boy and",0
2,"<bos>'The Adventures Of Barry McKenzie' started life as a satirical comic strip in 'Private Eye', written by Barry Humphries and based on an idea by Peter Cook. McKenzie ( 'Bazza' to his friends ) is a lanky, loud, hat-wearing Australian whose two main interests in life are sex ( despite never having had any ) and Fosters lager. In 1972, he found his way to the big screen for the first of two outings. It must have been tempting for Humphries to cast himself as 'Bazza', but he wisely left the job to Barry Crocker ( later to sing the theme to the television soap opera 'Neighbours'! ). Humphries instead played multiple roles in true Peter Sellers fashion, most notably Bazza's overbearing Aunt 'Edna Everage' ( this was before she became a Dame ).<br /><br />You know this is not going to be 'The Importance Of Being Ernest' when its",1
3,"<bos>Who are these ""They""- the actors? the filmmakers? Certainly couldn't be the audience- this is among the most air-puffed productions in existence. It's the kind of movie that looks like it was a lot of fun to shoot TOO much fun, nobody is getting any actual work done, and that almost always makes for a movie that's no fun to watch.<br /><br />Ritter dons glasses so as to hammer home his character's status as a sort of doppleganger of the bespectacled Bogdanovich; the scenes with the breezy Ms. Stratten are sweet, but have an embarrassing, look-guys-I'm-dating-the-prom-queen feel to them. Ben Gazzara sports his usual cat's-got-canary grin in a futile attempt to elevate the meager plot, which requires him to pursue Audrey Hepburn with all the interest of a narcoleptic at an insomnia clinic. In the meantime, the budding couple's respective children (nepotism alert: Bogdanovich's daughters) spew cute and pick up",0


## fastai IMDb data

When your dataset is saved in local folders, for example in a `split/label/file` layout, you can use `TextDataLoaders.from_folder` to create dataloaders.

In [ ]:
imdb_path = untar_data(URLs.IMDB)
Path.BASE_PATH = imdb_path


In [ ]:
imdb_path.ls()

[Path('test'), Path('tmp_clas'), Path('imdb.vocab'), Path('unsup'), Path('README'), Path('tmp_lm'), Path('train')]

In [ ]:
(imdb_path/'test').ls()

[Path('test/neg'), Path('test/pos'), Path('test/labeledBow.feat')]

In this case, we use `from_folder` and pass the root folder containing the dataset. IMDb does not have a validation folder, so we use the test folder as validation.

In [ ]:
folder_clas = TextDataLoadersNew.from_folder(imdb_path, valid='test',
    bs=4, seq_len=32)

In [ ]:
folder_clas.show_batch()

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>This is not a film you can really analyse separately from it's production. The audience became the film-makers to an extent unprecedented in the history of the American film industry; we felt so involved that viewing it becomes like watching the work of a friend. How is it possible to be objective? This is our movie, isn't it? Or is it? There may be nothing more disingenuous than a film-maker who promotes himself as the audience's friend, giving them all the naughty treats that the nannyish critics would deny them. Just look at that prime self-publicist Eli Roth, promising gore-hounds all the viscera missing from literally gutless mainstream horror films, only to churn out a watered down and technically incompetent piece of work like 'Hostel'.<br /><br />David R. Ellis may not have spawned the monster that was the internet response to his film, but he was, quite understandably, quick to",neg
2,"<bos>What happened? Those were the first words to come to mind after this awful movie finished for the first and last time on my computer screen. Nightmare on Elm St. had gone noticeably downhill after it's cult-classic of a first film, but I doubt anybody expected this horrible aberration. Nobody expected this cosmic joke of a film, and nobody is more distraught about it than I am.<br /><br />This is by far the worst ANOES film of the lot. It doesn't seem too bad at the beginning, with a genuinely creepy intro and a rather elongated shower scene featuring Alice. But then we hit rock bottom right at the beginning with bad acting and a jumbled sequence of events. I mean, sure, Freddy movies are supposed to be dreamlike and creepy, but this one is like a train-wreck in it's poor sequencing of events and awful plot setup. It feels",neg
3,"<bos>This movie starred a totally forgotten star from the 1930s, Jack Pearl (radio's ""Baron Munchausen"") as well as Jimmy Durante. However, 7-1/2 decades later, it's being billed as a Three Stooges film because they are the only ones in the film who the average person would recognize today. Film fanatics will also recognize the wonderful Edna May Oliver as well as Zazu Pitts.<br /><br />As for the Stooges, this is a film from there very early days--before MGM had any idea what to do with the team. At this point, they were known as ""Ted Healy and his Stooges"" as Healy was the front man. Fortunately for the Stooges, they soon left this nasty and rather untalented man (read up on him--you'll see what I mean) and the rest is history. Within a year, they were making very successful shorts for Columbia and executives at MGM were soon kicking themselves",neg


## Non-standard data

In this section, we will see how to build dataloaders when your dataset doesn't follow the standard layouts that the factory methods support.

The example below uses a pandas DataFrame, but it can work with any kind of dataset. For example, you could have documents in a folder and labels in a CSV. As long as you can turn your data into a list of documents and labels (or just documents for a language model), you can create dataloaders.

In [ ]:
records = pd.DataFrame([{'body': 'The soup needs more garlic.', 'topic': 'cooking'},
    {'body': 'The team scored twice.', 'topic': 'sport'},
    {'body': 'Bake the bread until golden.', 'topic': 'cooking'},
    {'body': 'The goalkeeper saved the final shot.', 'topic': 'sport'}])
records

,body,topic
0,The soup needs more garlic.,cooking
1,The team scored twice.,sport
2,Bake the bread until golden.,cooking
3,The goalkeeper saved the final shot.,sport


We first extract the texts and labels from the DataFrame.

In [ ]:
texts = records['body'].tolist()
labels = records['topic'].tolist()


We then tokenize the texts. `tokenize_texts` returns an Arrow column with one list of token IDs per document. In the case below, the first list holds the 6 tokens of the first document, the second list holds the 5 tokens of the second document, and so on.

In [ ]:
doc_tok = hf_tokenizer()
ids = tokenize_texts(texts, doc_tok)
ids

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

<pyarrow.lib.ChunkedArray object>
[
  [
    [
      464,
      17141,
      2476,
      517,
      17644,
      13
    ],
    [
      464,
      1074,
      7781,
      5403,
      13
    ],
    [
      33,
      539,
      262,
      8509,
      1566,
      10861,
      13
    ],
    [
      464,
      26318,
      7448,
      262,
      2457,
      2823,
      13
    ]
  ]
]

`TokenDocs` is a helper class that lets us access each document by index as a NumPy array. For example, `docs[0]` contains the tokens of the first document.

In [ ]:
docs = TokenDocs(ids)
docs[0]

array([  464, 17141,  2476,   517, 17644,    13], dtype=uint16)

We create the dataloaders from our `TokenDocs` and we specify which docs are used for training and validation using `doc_splits`.

In [ ]:
doc_splits = ([0, 1], [2, 3])
memory_clas = TextDataLoadersNew.from_docs(docs, doc_tok, doc_splits, labels=labels, bs=2, seq_len=8)
test_eq(memory_clas.vocab[1], ['cooking', 'sport'])
memory_clas.show_batch(max_n=2, trunc_at=30)

,text,category
0,<bos>The soup needs more garlic.<eos>,cooking
1,<bos>The team scored twice.<eos>,sport


## A small ULMFiT training run

In this section, we’ll try a small ULMFiT training run with three stages:

- Stage 1: Train a language model (LM) from scratch on WikiText.
- Stage 2: Fine-tune that LM on the target IMDb texts.
- Stage 3: Fine-tune a classifier using the encoder from stage 2.

This section focuses only on showing how the new dataloaders API fits into training an NLP model, keeping the training minimal. It doesn’t follow the exact ULMFiT recipe.


The full ULMFiT recipe trains a forward and a backward model. The final model is the ensemble of the two. As a demo, we train the backward model for stage 1 only. Stages 2 and 3 follow the same pattern, using backward dataloaders.

### Stage 1: LM from WikiText

Here we create language model dataloaders from the WikiText-103 dataset, as we showed above. We then pass them to `language_model_learner` and train the model.

In [ ]:
wiki_train = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:30]', valid='validation[:20]',
    bs=4, seq_len=32)

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [ ]:
lm_config = dict(awd_lstm_lm_config, pad_token=wiki_train.pad_id)
learn = language_model_learner(wiki_train, AWD_LSTM, config=lm_config, pretrained=False,
    metrics=[accuracy, Perplexity()])
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         9.422750    8.642236    0.039605  5665.983398 00:02     


### Stage 2: Adapt WikiText LM to IMDb

In this section, we create new language model dataloaders from the IMDb dataset, then replace the `Learner` dataloaders from stage 1 to train with these new ones.

In [ ]:
imdb_train_lm = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text',
    train='train[:8]+unsupervised[:8]', valid='test[:4]',
    bs=4, seq_len=32)
learn.dls = imdb_train_lm

learn.fit(1)
learn.save_encoder('imdb-forward')

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         8.569011    8.093412    0.037079  3272.836914 00:01     


### Stage 3: Classification

In this final stage, we create new classification dataloaders from the IMDb dataset, create a new learner, load the encoder from stage 2, and fine-tune the classifier.

In [ ]:
imdb_train_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:8]+train[-8:]', valid='test[:4]+test[-4:]',
    bs=4, seq_len=32)

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [ ]:
clas_config = dict(awd_lstm_clas_config, pad_token=imdb_train_clas.pad_id)
learn = text_classifier_learner(imdb_train_clas, AWD_LSTM, config=clas_config, seq_len=32,
    pretrained=False, metrics=accuracy)
learn.load_encoder('imdb-forward')
learn.unfreeze()
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  time    
0         0.726990    0.683889    0.625000  00:02     


In [ ]:
learn.show_results()

,text,category,category_
0,"<bos>STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morning <br /><br />Former New Orleans homicide cop Jack Robideaux (Jean Claude Van Damme) is re-assigned to Columbus, a small but violent town in Mexico to help the police there with their efforts to stop a major heroin smuggling operation into their town. The culprits turn out to be ex-military, lead by former commander Benjamin Meyers (Stephen Lord, otherwise known as Jase from East Enders) who is using a special method he learned in Afghanistan to fight off his opponents. But Jack has a more personal reason for taking him down, that draws the two men into an explosive final showdown where only one will walk away alive.<br /><br />After Until Death, Van Damme appeared to be on a high, showing he could make the best straight to video films in the action market.",0,1
1,"<bos>Five minutes in, i started to feel how naff this was looking, you've got a completely unheroic hero and his overweight fool of a friend. Seen it all before, yeah right. I was getting ready to be bored out of my mind for a good few hours. This is something i have become quite used to... haven't we all. Then after a few minutes of testosterone fuelled insults and such, the truck appeared. Okay the filming techniques used to make it look fast were clumsy, but who cares! That truck is amazing! Soon however that is taken away again and we're back to the geek and his overweight friend. But now i'm satisfied that at least it won't be too terrible. I then proceed to be amazed again and again by the cleverness of the film. There are so many jokes at their expense, it's like everyone in the world",1,1
2,"<bos>I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn't match the background, and painfully one-dimensional characters cannot be overcome with a 'sci-fi' setting. (I'm sure there are those of you out there who think Babylon 5 is good sci-fi TV. It's not. It's clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It's really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are",0,0
3,"<bos>Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All of this is entertaining and competently handled but there is nothing that really blows you away if you've seen your share before.<br /><br />The plot is made interesting by the inclusion of a rabbit, which is clever but hardly profound. Many of the characters are heavily stereotyped -- the angry veterans, the terrified illegal aliens, the crooked cops, the indifferent feds, the bitchy tough lady station head, the crooked politician, the fat federale who looks like he was typecast as the Mexican in a Hollywood movie from the 1940s. All passably acted but again nothing special.<br /><br />I thought the main villains were pretty well done and fairly",0,0


### Backward

Ensembling forward and backward models, where the backward model is trained on text in reverse order, can improve results according to the ULMFiT paper.

We can train the backward model by passing `backwards=True` when we create the dataloaders. Here’s an example of stage 1 using backward dataloaders.

In [ ]:
wiki_backward = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:30]', valid='validation[:20]',
    backwards=True, bs=4, seq_len=32)

In [ ]:
wiki_backward.show_batch()

 written by Seiko Fujibayashi , who had worked on May 'n on previous singles . 
 = = = Release = = = 
<eos>
 handled by Media.Vision. The original scenario was written Kazuki Yamanobe , while the script was written by Hiroyuki Fujii , Koichi Maj
 incursion into Imperial territory , other orders cause certain members of the 422nd great distress . One such member , Gusurg , becomes so enraged that he aband
alkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player
 were written by Seiko Fujibayashi , who had worked on May 'n on previous singles . 
 = = = Release = = = 

 also handled by Media.Vision. The original scenario was written Kazuki Yamanobe , while the script was written by Hiroyuki Fujii , Koichi
 successful incursion into Imperial territory , other orders cause certain members of the 422nd great distress . One such member , Gusurg , becomes so enraged that he ab
 Valkyira C

In [ ]:
wiki_backward.tfms

Pipeline: HfTokenizer -> Reverse

In [ ]:
learn = language_model_learner(wiki_backward, AWD_LSTM, config=lm_config, pretrained=False,
    metrics=[accuracy, Perplexity()])
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         9.420747    8.705386    0.046441  6035.332031 00:00     
